# Lesson 03 - Microsoft Agent Framework: Advanced Deep Dive

## Domain: Hospital Patient Management Agent

In this lesson we build a more advanced agent for a hospital scenario. The agent can:
- Look up patient records
- Check doctor availability
- Book appointments
- Retrieve lab results

---

##  The Four Pillars of Microsoft Agent Framework

| Pillar | Class / Decorator | Role |
|--------|------------------|------|
| **Client** | `FoundryChatClient` | Connects to the Azure AI model endpoint |
| **Agent** | `provider.as_agent()` | Wraps client with a system prompt + tools |
| **Tools** | `@tool` decorator | Python functions the agent can invoke |
| **Session** | `agent.create_session()` | Stores conversation history for multi-turn chat |

```
┌─────────────────────────────────────────────┐
│              Microsoft Agent Framework       │
│                                             │
│   User Message                              │
│        │                                    │
│        ▼                                    │
│   ┌─────────┐    ┌──────────────────────┐   │
│   │ SESSION │───▶│       AGENT          │   │
│   │(History)│    │  (Instructions +     │   │
│   └─────────┘    │   Tools + Name)      │   │
│                  └──────────┬───────────┘   │
│                             │               │
│                             ▼               │
│                  ┌──────────────────────┐   │
│                  │       CLIENT         │   │
│                  │ (Azure AI Endpoint)  │   │
│                  └──────────┬───────────┘   │
│                             │               │
│              ┌──────────────┼───────────┐   │
│              ▼              ▼           ▼   │
│          ┌───────┐   ┌──────────┐  ┌──────┐ │
│          │ Tool1 │   │  Tool2   │  │Tool3 │ │
│          │(look- │   │ (check   │  │(book │ │
│          │ up    │   │  doctor) │  │appt) │ │
│          │patient│   └──────────┘  └──────┘ │
│          └───────┘                          │
└─────────────────────────────────────────────┘
```

---
## 1 Setup — Install & Import

In [1]:
# Install required packages
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from datetime import datetime, timedelta
from typing import Annotated

# MAF core imports
from agent_framework import tool           # ← converts a Python function into an agent tool
from agent_framework.foundry import FoundryChatClient  # ← the CLIENT
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

def extract_text(response) -> str:
    """Safely pull a plain string from whatever agent.run() returns."""
    if response is None:
        return "(no response)"
    if isinstance(response, str):
        return response.strip() or "(empty string)"
    if hasattr(response, "output_text"):
        return str(response.output_text).strip()
    if hasattr(response, "content") and isinstance(response.content, str):
        return response.content.strip()
    if hasattr(response, "content") and isinstance(response.content, list):
        parts = []
        for block in response.content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            elif hasattr(block, "text"):
                parts.append(str(block.text))
        return " ".join(parts).strip() or "(empty content list)"
    if hasattr(response, "text"):
        return str(response.text).strip()
    return f"(type={type(response).__name__}) {repr(response)[:300]}"

print("✅ Imports successful + extract_text() helper ready")


C:\Users\Janu\AppData\Roaming\Python\Python311\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\Janu\AppData\Roaming\Python\Python311\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Imports successful + extract_text() helper ready


---
##  PILLAR 1 — Client

The **Client** is responsible for:
- Authenticating with Azure
- Sending requests to the model endpoint
- Receiving and parsing model responses

> Think of the Client as the **phone line** — it connects your code to the AI model.

```
Your Code  ──→  FoundryChatClient  ──→  Azure OpenAI Model
```

In [3]:
# ─── PILLAR 1: CLIENT ─────────────────────────────────────────────────────────
#
# FoundryChatClient needs:
#   project_endpoint  — the Azure AI Foundry project URL
#   model             — the deployment name (e.g. "gpt-4o")
#   credential        — Azure auth method (AzureCliCredential uses `az login`)
#
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model    = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing environment variables.\n"
        "Set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME "
        "in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

print(f"✅ Client created — connected to model: {model}")

✅ Client created — connected to model: gpt-4.1


---
## 3️⃣ PILLAR 3 — Tools  *(defined before the Agent so we can pass them in)*

Tools are ordinary Python functions decorated with `@tool`. The framework:
1. Reads the **docstring** → becomes the tool description sent to the model
2. Reads **`Annotated[type, "desc"]`** → tells the model what each parameter means
3. Handles **calling the function** when the model decides to use it

### `approval_mode` options

| Value | Behaviour | Use in Jupyter? |
|-------|-----------|----------------|
| `"never_require"` | Runs automatically | ✅ Always safe |
| `"always_require"` | Framework pauses for CLI confirmation | ❌ Breaks in Jupyter |
| `"auto"` | Framework decides | ⚠️ Risky in Jupyter |

> **Rule:** In Jupyter, always use `never_require`. If you need human approval,
> ask with `input()` **before** calling `agent.run()` — never inside the tool.

We define **4 tools** for our hospital agent:


In [ ]:
# ─── TOOL 1: Look up a patient ────────────────────────────────────────────────

# Simulated patient database
PATIENTS_DB = {
    "P001": {"name": "Alice Nair",    "age": 34, "condition": "Hypertension",  "doctor": "Dr. Perera"},
    "P002": {"name": "Bob Fernando",  "age": 56, "condition": "Diabetes",       "doctor": "Dr. Silva"},
    "P003": {"name": "Clara Mendis",  "age": 28, "condition": "Asthma",         "doctor": "Dr. Perera"},
    "P004": {"name": "David Raj",     "age": 72, "condition": "Arthritis",      "doctor": "Dr. Gunasekara"},
}

@tool(approval_mode="never_require")
def get_patient_info(
    patient_id: Annotated[str, "The unique patient ID, e.g. P001"]
) -> str:
    """
    Retrieve basic information about a hospital patient by their ID.
    Returns name, age, current condition, and assigned doctor.
    """
    patient = PATIENTS_DB.get(patient_id.upper())
    if not patient:
        return f"No patient found with ID '{patient_id}'."
    return (
        f"Patient {patient_id}: {patient['name']}, Age {patient['age']}, "
        f"Condition: {patient['condition']}, Assigned Doctor: {patient['doctor']}"
    )

print("Tool 1 defined: get_patient_info")

✅ Tool 1 defined: get_patient_info


In [ ]:
# ─── TOOL 2: Check doctor availability ───────────────────────────────────────

DOCTOR_SCHEDULE = {
    "Dr. Perera":      {"available_days": ["Monday", "Wednesday", "Friday"],  "slots": 4},
    "Dr. Silva":       {"available_days": ["Tuesday", "Thursday"],             "slots": 6},
    "Dr. Gunasekara":  {"available_days": ["Monday", "Tuesday", "Thursday"],   "slots": 3},
}

@tool(approval_mode="never_require")
def check_doctor_availability(
    doctor_name: Annotated[str, "Full name of the doctor, e.g. Dr. Perera"],
    day:         Annotated[str, "Day of the week to check, e.g. Monday"]
) -> str:
    """
    Check whether a specific doctor is available on a given day of the week
    and how many open appointment slots remain.
    """
    schedule = DOCTOR_SCHEDULE.get(doctor_name)
    if not schedule:
        return f"Doctor '{doctor_name}' not found in the system."
    if day.capitalize() in schedule["available_days"]:
        return (
            f"{doctor_name} is available on {day} "
            f"with {schedule['slots']} open slots."
        )
    return f"{doctor_name} is NOT available on {day}."

print(" Tool 2 defined: check_doctor_availability")

✅ Tool 2 defined: check_doctor_availability


In [ ]:
# ─── TOOL 3: Book an appointment ─────────────────────────────────────────────
#
# ✅ CORRECT PATTERN: Keep the tool simple (never_require).
#    Never put input() inside a tool — the framework sends the tool call
#    to the model and expects an immediate return value. Any blocking call
#    (input, sleep, file dialog) prevents the return, so the model never
#    gets the tool output → BadRequestError: 'No tool output found'.
#
#    Human approval belongs OUTSIDE agent.run() — see Turn 4 cell.

APPOINTMENTS = []  # in-memory store of booked appointments

@tool(approval_mode="never_require")   # ← always use this in Jupyter
def book_appointment(
    patient_id:  Annotated[str, "Patient ID, e.g. P001"],
    doctor_name: Annotated[str, "Full name of the doctor"],
    day:         Annotated[str, "Day of the week for the appointment"]
) -> str:
    """
    Book a hospital appointment for a patient with a specific doctor on a given day.
    Records the appointment in the system and returns a confirmation.
    """
    patient = PATIENTS_DB.get(patient_id.upper())
    if not patient:
        return f"Cannot book: patient '{patient_id}' not found."

    schedule = DOCTOR_SCHEDULE.get(doctor_name)
    if not schedule or day.capitalize() not in schedule["available_days"]:
        return f"Cannot book: {doctor_name} is not available on {day}."

    appointment = {
        "id":      f"APT{len(APPOINTMENTS)+1:03d}",
        "patient": patient["name"],
        "doctor":  doctor_name,
        "day":     day.capitalize(),
    }
    APPOINTMENTS.append(appointment)
    return (
        f"Appointment booked! ID: {appointment['id']} — "
        f"{appointment['patient']} with {doctor_name} on {day.capitalize()}."
    )

print(" Tool 3 defined: book_appointment")


✅ Tool 3 defined: book_appointment


In [ ]:
# ─── TOOL 4: Get lab results ──────────────────────────────────────────────────

LAB_RESULTS = {
    "P001": {"test": "Blood Pressure",  "result": "140/90 mmHg",  "status": "High — follow-up required"},
    "P002": {"test": "HbA1c",           "result": "7.8%",          "status": "Elevated — medication review advised"},
    "P003": {"test": "Peak Flow",       "result": "320 L/min",     "status": "Below normal — inhaler prescribed"},
    "P004": {"test": "ESR",             "result": "45 mm/hr",      "status": "Slightly elevated — monitor"},
}

@tool(approval_mode="never_require")
def get_lab_results(
    patient_id: Annotated[str, "The patient ID to retrieve lab results for"]
) -> str:
    """
    Retrieve the most recent laboratory test results for a patient.
    Returns the test name, numeric result, and clinical status.
    """
    result = LAB_RESULTS.get(patient_id.upper())
    if not result:
        return f"No lab results found for patient '{patient_id}'."
    return (
        f"Lab results for {patient_id} — "
        f"Test: {result['test']}, Result: {result['result']}, "
        f"Status: {result['status']}"
    )

print("Tool 4 defined: get_lab_results")
print()
print("All 4 tools ready:", [
    "get_patient_info",
    "check_doctor_availability",
    "book_appointment",
    "get_lab_results",
])

✅ Tool 4 defined: get_lab_results

All 4 tools ready: ['get_patient_info', 'check_doctor_availability', 'book_appointment', 'get_lab_results']


---
## 4️⃣ PILLAR 2 — Agent

The **Agent** is the core orchestrator. It combines:
- The **client** (model connection)
- A **name** (identity)
- **Instructions** (system prompt — defines persona and rules)
- A list of **tools** (what actions it can take)

> Think of the Agent as a **specialist employee**: you give them a job title, a rulebook, and a set of tools. The client is just their phone.

### ✏️ Writing Good Instructions
- Be explicit about the agent's **role** and **scope**
- State **what to do** and **what NOT to do**
- Tell the agent when to use which tool

In [ ]:
# ─── PILLAR 2: AGENT ──────────────────────────────────────────────────────────

SYSTEM_INSTRUCTIONS = """
You are HospitalAssist, an AI agent for St. Lakeside Hospital.

Your responsibilities:
1. Look up patient information when a patient ID is mentioned.
2. Check doctor availability before suggesting or confirming appointments.
3. Book appointments only after confirming the doctor is available on the requested day.
4. Retrieve lab results when asked about test outcomes or health status.

Rules:
- Always verify patient ID format (P001, P002, etc.) before tool calls.
- Never make up patient data — always use the provided tools.
- If unsure, ask a clarifying question before calling a tool.
- Be concise, professional, and empathetic in your responses.
"""

agent = provider.as_agent(
    name="HospitalAssist",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[
        get_patient_info,
        check_doctor_availability,
        book_appointment,
        get_lab_results,
    ]
)

print("Agent 'HospitalAssist' created with 4 tools")

✅ Agent 'HospitalAssist' created with 4 tools


---
## 5️⃣ PILLAR 4 — Session (Multi-Turn Conversation)

A **Session** stores the full conversation history. Without it, every `agent.run()` call would forget what was said before — like starting a fresh phone call each time.

```
Turn 1:  User: "Look up patient P001"        → Agent calls get_patient_info
Turn 2:  User: "Is their doctor free Monday?" → Agent REMEMBERS P001's doctor from Turn 1
Turn 3:  User: "Book it"                      → Agent REMEMBERS doctor + day from Turn 2
```

This **context carry-over** is what makes the conversation feel natural.

In [ ]:
# ─── PILLAR 4: SESSION ────────────────────────────────────────────────────────

session = agent.create_session()
print("Session created — conversation history is now being tracked")

✅ Session created — conversation history is now being tracked


---
## 6️⃣ Running the Multi-Turn Conversation

Watch how the agent:
- Decides **which tool to call** based on the user's message
- **Carries context** between turns using the session
- Handles a booking that **requires approval** before executing

In [ ]:
# ── Turn 1: Look up a patient ─────────────────────────────────────────────────
# Agent will call: get_patient_info(patient_id="P002")

response = await agent.run(
    "Can you pull up the details for patient P002?",
    session=session,
)
text = extract_text(response)
print(f"Agent: {text}")


🤖 Agent: Patient P002 is Bob Fernando, age 56. He is being treated for diabetes, and his assigned doctor is Dr. Silva.

If you need more specific information, such as recent lab results or upcoming appointments, please let me know.


In [ ]:
# ── Turn 2: Ask about lab results (agent remembers P002 from Turn 1) ──────────
# Agent will call: get_lab_results(patient_id="P002")
# Notice: user didn't repeat the patient ID — the session carries that context!

response = await agent.run(
    "What are their latest lab results?",
    session=session,
)
text = extract_text(response)
print(f"Agent: {text}")


🤖 Agent: Bob Fernando's latest lab result is for the HbA1c test, with a result of 7.8%. This is considered elevated, and a medication review is advised.

Would you like to discuss these results with Dr. Silva or schedule a follow-up appointment?


In [ ]:
# ── Turn 3: Check doctor availability ────────────────────────────────────────
# Agent will call: check_doctor_availability(doctor_name="Dr. Silva", day="Tuesday")
# Agent knows P002's doctor is Dr. Silva (from Turn 1)

response = await agent.run(
    "Is Dr. Silva available on Tuesday?",
    session=session,
)
text = extract_text(response)
print(f" Agent: {text}")


🤖 Agent: Yes, Dr. Silva is available on Tuesday and has 6 open appointment slots.

Would you like to book an appointment for Bob Fernando with Dr. Silva on Tuesday?


In [ ]:
# ── Turn 4: Human approval BEFORE agent.run() ───────────────────────────────
#
# KEY LESSON: Approval must happen OUTSIDE agent.run(), not inside the tool.
#
# WHY:  agent.run() → model decides to call book_appointment → framework
#       calls the Python function → function must return immediately.
#       Any blocking call inside the tool (input, sleep) prevents the return
#       → model never gets the tool result → BadRequestError.
#
# CORRECT PATTERN: Ask the user BEFORE calling agent.run().
#   If user says no  → skip the agent call entirely.
#   If user says yes → call agent.run() and the tool runs freely.

print("⚠️  You are about to book an appointment.")
print("   Patient : P002 — Bob Fernando")
print("   Doctor  : Dr. Silva")
print("   Day     : Tuesday")
confirm = input("\nConfirm? (yes / no): ").strip().lower()

if confirm in ("yes", "y"):
    response = await agent.run(
        "Please book an appointment for patient P002 with Dr. Silva on Tuesday.",
        session=session,
    )
    text = extract_text(response)
    print(f"\n🤖 Agent: {text}")
else:
    print("Booking cancelled — agent.run() was never called.")


⚠️  You are about to book an appointment.
   Patient : P002 — Bob Fernando
   Doctor  : Dr. Silva
   Day     : Tuesday

🤖 Agent: The appointment for Bob Fernando (patient P002) with Dr. Silva on Tuesday has been successfully booked.

If you need the appointment details or further assistance, please let me know.


In [14]:
# ── Turn 5: New patient, new tool chain ──────────────────────────────────────
# Watch the agent chain TWO tool calls: get_patient_info → get_lab_results

response = await agent.run(
    "Now check on patient P001 and tell me their condition and lab results.",
    session=session,
)
text = extract_text(response)
print(f"🤖 Agent: {text}")


🤖 Agent: Patient P001 is Alice Nair, age 34. Her condition is hypertension, and her assigned doctor is Dr. Perera.

Her latest lab result shows a blood pressure reading of 140/90 mmHg, which is considered high. A follow-up is required.

Would you like to arrange an appointment with Dr. Perera or need further details?


---
## 7️⃣ How Tool Calling Works Internally

Understanding what happens under the hood makes you a better agent developer:

```
Step 1  User sends:  "What are the lab results for P002?"
             │
             ▼
Step 2  Agent sends to model:
         ┌─ System prompt (instructions)
         ├─ Full conversation history (session)
         └─ Tool schemas (name + description + parameters)
             │
             ▼
Step 3  Model responds with a TOOL CALL (not plain text):
         { "tool": "get_lab_results", "args": { "patient_id": "P002" } }
             │
             ▼
Step 4  Framework executes the Python function → gets result string
             │
             ▼
Step 5  Framework sends the tool result BACK to the model
             │
             ▼
Step 6  Model generates a natural language reply using the result
             │
             ▼
Step 7  User sees the final answer
```

The model never executes Python — it only **decides** which tool to call. The framework does the execution.

---
## 8️⃣ Bonus — Running Without a Session (Stateless)

For comparison, here is a **stateless** call — no session, no history. Each call is independent. Use this for one-shot tasks that don't need context.

In [ ]:
# ── Stateless call — no session passed ───────────────────────────────────────
# Each call here is completely independent — agent has no memory of prior turns.

response = await agent.run(
    "Look up patient P004 and check if Dr. Gunasekara is available on Monday."
    # No session= argument ← stateless!
)
text = extract_text(response)
print(f"Agent (stateless): {text}")


🤖 Agent (stateless): Patient P004 is David Raj, age 72, currently being treated for arthritis by Dr. Gunasekara.

Dr. Gunasekara is available on Monday with 3 open appointment slots. Would you like to book an appointment for David Raj on that day?


---
## Summary

You have now seen all four MAF pillars working together in a real-world scenario:

| Pillar | What We Built | Key Takeaway |
|--------|--------------|---------------|
| **Client** | `FoundryChatClient` connected to Azure OpenAI | Handles auth + model communication |
| **Agent** | `HospitalAssist` with 4 tools + system prompt | Bundles client + instructions + tools |
| **Tools** | `get_patient_info`, `check_doctor_availability`, `book_appointment`, `get_lab_results` | Python functions the model can call; use `approval_mode` to gate sensitive actions |
| **Session** | Created once, passed to every `agent.run()` | Maintains full conversation history; enables context carry-over across turns |

### 🔑 Key Concepts to Remember

- **The model never runs Python** — it only outputs a tool call request; the framework executes it.
- **`approval_mode="always_require"`** breaks in Jupyter — use a manual `input()` guard inside the tool instead.
- **Session = memory** — without it, every turn is a fresh conversation.
- **Docstrings + `Annotated` types** = the model's understanding of your tools. Write them clearly!
- **Instructions quality matters** — the clearer your system prompt, the more predictable your agent's behaviour.

---

```
Client  ──▶  Agent  ──▶  Tool1: get_patient_info          (auto)
                    ──▶  Tool2: check_doctor_availability  (auto)
                    ──▶  Tool3: book_appointment           (manual input() guard ⚠️)
                    ──▶  Tool4: get_lab_results            (auto)
Session ──▶  [Turn1, Turn2, Turn3 ...]
```